In [4]:
import os
import re
import shutil
import random
from collections import defaultdict
import torch
import torchvision
import rasterio
import geopandas as gpd
from shapely.geometry import box
import numpy as np
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models import ResNet50_Weights
from tqdm import tqdm

# --- CONFIGURACIÓN ---
SOURCE_IMG_DIR = r"D:\Silos\Base de datos\procesado_sin_nubes\tiles"
SOURCE_SHP_DIR = r"D:\Silos\Base de datos\procesado_sin_nubes\tiles\indices_INBNV\masks\polygons"
MODEL_PATH = r"D:\Silos\modelo_silos_v3_aug.pth"
OUTPUT_DIR = r"D:\Silos\Golden_Dataset_1000"
SAMPLE_SIZE = 1000
THRESHOLD = 0.4
SEED = 123

YEARS_TO_INCLUDE = [2020, 2021, 2022, 2023, 2024, 2025]
MONTHS_TO_INCLUDE = [4, 5, 6, 7, 8]
MIN_TILE_DISTANCE = 2


def parse_tile_name(filename):
    m = re.match(r'(\d{4})_(\d{1,2})_\d{1,2}_\d+_tile_r(\d+)_c(\d+)', filename)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
    return None


def stratified_sample(all_tifs, sample_size, seed, years, months, min_dist):
    random.seed(seed)
    
    parsed = []
    for f in all_tifs:
        info = parse_tile_name(f)
        if info is None:
            continue
        year, month, row, col = info
        if year not in years or month not in months:
            continue
        parsed.append({'filename': f, 'year': year, 'month': month, 'row': row, 'col': col})
    
    print(f"Tiles válidos (años {years}, meses {months}): {len(parsed)}")
    
    groups = defaultdict(list)
    for item in parsed:
        key = (item['year'], item['month'])
        groups[key].append(item)
    
    print(f"Estratos año-mes encontrados: {len(groups)}")
    for k, v in sorted(groups.items()):
        print(f"  {k[0]}-{k[1]:02d}: {len(v)} tiles disponibles")
    
    total_available = sum(len(v) for v in groups.values())
    per_stratum = {}
    remainder = sample_size
    
    for key in sorted(groups.keys()):
        n_available = len(groups[key])
        ideal = max(1, int(sample_size * n_available / total_available))
        per_stratum[key] = min(ideal, n_available)
        remainder -= per_stratum[key]
    
    if remainder > 0:
        for key in sorted(groups.keys(), key=lambda k: len(groups[k]), reverse=True):
            can_add = len(groups[key]) - per_stratum[key]
            add = min(can_add, remainder)
            per_stratum[key] += add
            remainder -= add
            if remainder <= 0:
                break
    
    print(f"\nPlan de muestreo (target {sample_size}):")
    for k in sorted(per_stratum.keys()):
        print(f"  {k[0]}-{k[1]:02d}: {per_stratum[k]} de {len(groups[k])} disponibles")
    
    selected = []
    occupied_positions = set()
    
    for key in sorted(groups.keys()):
        candidates = groups[key].copy()
        random.shuffle(candidates)
        
        n_target = per_stratum[key]
        stratum_selected = []
        
        for item in candidates:
            if len(stratum_selected) >= n_target:
                break
            
            r, c = item['row'], item['col']
            too_close = False
            for (oy, om, orow, ocol) in occupied_positions:
                if oy == item['year'] and om == item['month']:
                    if abs(r - orow) < min_dist and abs(c - ocol) < min_dist:
                        too_close = True
                        break
            
            if not too_close:
                stratum_selected.append(item)
                occupied_positions.add((item['year'], item['month'], r, c))
        
        selected.extend(stratum_selected)
    
    print(f"\nTotal seleccionado tras filtro espacial: {len(selected)}")
    return [item['filename'] for item in selected]


def get_model(num_classes):
    anchor_sizes = ((16,), (32,), (64,), (128,), (256,))
    aspect_ratios = ((0.2, 0.5, 1.0, 2.0, 5.0),) * len(anchor_sizes)
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=ResNet50_Weights.DEFAULT,
        rpn_anchor_generator=anchor_generator
    )
    model.roi_heads.box_predictor = FastRCNNPredictor(
        model.roi_heads.box_predictor.cls_score.in_features, num_classes
    )
    return model


def preprocess(img_path):
    with rasterio.open(img_path) as src:
        try:
            img = src.read([1, 2, 3])
        except:
            img = src.read()[:3]
    img = np.nan_to_num(img.astype(np.float32), nan=0.0)
    if np.max(img) > 255.0:
        img /= 10000.0
    elif np.max(img) > 1.0:
        img /= 255.0
    return torch.as_tensor(np.clip(img, 0, 1), dtype=torch.float32)


def main_golden():
    print(f"--- CREANDO GOLDEN DATASET (ESTRATIFICADO) ---")
    
    device = torch.device('cpu')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("Cargando Modelo V3...")
    if not os.path.exists(MODEL_PATH):
        print("ERROR: No encuentro el modelo V3.")
        return
    model = get_model(2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()
    
    print("Listando imágenes fuente...")
    all_tifs = [f for f in os.listdir(SOURCE_IMG_DIR) if f.lower().endswith('.tif')]
    print(f"Total de tiles en disco: {len(all_tifs)}")
    
    selection = stratified_sample(
        all_tifs, SAMPLE_SIZE, SEED,
        years=YEARS_TO_INCLUDE,
        months=MONTHS_TO_INCLUDE,
        min_dist=MIN_TILE_DISTANCE
    )
    
    with open(os.path.join(OUTPUT_DIR, "selection_log.txt"), "w") as f:
        f.write(f"Seed: {SEED}\n")
        f.write(f"Years: {YEARS_TO_INCLUDE}\n")
        f.write(f"Months: {MONTHS_TO_INCLUDE}\n")
        f.write(f"Min tile distance: {MIN_TILE_DISTANCE}\n")
        f.write(f"Total selected: {len(selection)}\n\n")
        for s in sorted(selection):
            f.write(s + "\n")
    
    # Detectar cuántos ya fueron procesados (para resumir tras un error)
    already_done = set()
    for f in os.listdir(OUTPUT_DIR):
        if f.startswith("pred_") and f.endswith(".shp"):
            already_done.add(f.replace("pred_", "").replace(".shp", "") + ".tif")
    
    pending = [t for t in selection if t not in already_done]
    print(f"\nYa procesados: {len(already_done)} | Pendientes: {len(pending)}")
    
    for tif_name in tqdm(pending, desc="Procesando pendientes"):
        base_name = os.path.splitext(tif_name)[0]
        
        src_tif_path = os.path.join(SOURCE_IMG_DIR, tif_name)
        dst_tif_path = os.path.join(OUTPUT_DIR, tif_name)
        dst_pred_shp_path = os.path.join(OUTPUT_DIR, f"pred_{base_name}.shp")
        
        # A) Copiar IMAGEN
        if not os.path.exists(dst_tif_path):
            shutil.copy2(src_tif_path, dst_tif_path)
        
        # B) Copiar GROUND TRUTH
        src_shp_exact = os.path.join(SOURCE_SHP_DIR, base_name + ".shp")
        match_base = None
        
        if os.path.exists(src_shp_exact):
            match_base = base_name
        else:
            candidates = [f for f in os.listdir(SOURCE_SHP_DIR)
                          if f.startswith(base_name) and f.endswith(".shp")]
            if candidates:
                match_base = os.path.splitext(candidates[0])[0]
        
        if match_base:
            for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
                src_file = os.path.join(SOURCE_SHP_DIR, match_base + ext)
                dst_file = os.path.join(OUTPUT_DIR, f"gt_{base_name}{ext}")
                if os.path.exists(src_file) and not os.path.exists(dst_file):
                    shutil.copy2(src_file, dst_file)
        
        # C) Generar PREDICCIÓN V3
        img_tensor = preprocess(src_tif_path)
        with torch.no_grad():
            prediction = model([img_tensor.to(device)])
        
        boxes = prediction[0]['boxes'].cpu().numpy()
        scores = prediction[0]['scores'].cpu().numpy()
        
        geometries = []
        valid_scores = []
        with rasterio.open(src_tif_path) as src:
            transform = src.transform
            crs = src.crs
            for b, s in zip(boxes, scores):
                if s >= THRESHOLD:
                    xmin, ymin, xmax, ymax = b
                    x_min, y_max = rasterio.transform.xy(transform, ymin, xmin, offset='ul')
                    x_max, y_min = rasterio.transform.xy(transform, ymax, xmax, offset='ul')
                    geometries.append(box(x_min, y_min, x_max, y_max))
                    valid_scores.append(float(s))
        
        if len(geometries) > 0:
            gdf = gpd.GeoDataFrame(
                {'id': range(len(geometries)), 'score': valid_scores},
                geometry=geometries, crs=crs
            )
            gdf.to_file(dst_pred_shp_path)
        else:
            gdf_empty = gpd.GeoDataFrame(
                {'id': [], 'score': []},
                geometry=[], crs=crs
            )
            gdf_empty.to_file(dst_pred_shp_path)

    print("\n" + "=" * 50)
    print(f"¡GOLDEN DATASET LISTO! -> {OUTPUT_DIR}")
    print(f"Total imágenes: {len(selection)} ({len(already_done)} previas + {len(pending)} nuevas)")
    print(f"Log de selección: selection_log.txt")
    print("=" * 50)


if __name__ == "__main__":
    main_golden()

--- CREANDO GOLDEN DATASET (ESTRATIFICADO) ---
Cargando Modelo V3...
Listando imágenes fuente...
Total de tiles en disco: 25749
Tiles válidos (años [2020, 2021, 2022, 2023, 2024, 2025], meses [4, 5, 6, 7, 8]): 25749
Estratos año-mes encontrados: 18
  2020-06: 2321 tiles disponibles
  2020-07: 1464 tiles disponibles
  2020-08: 1907 tiles disponibles
  2021-06: 2256 tiles disponibles
  2021-07: 2704 tiles disponibles
  2021-08: 3136 tiles disponibles
  2022-06: 1202 tiles disponibles
  2022-07: 873 tiles disponibles
  2022-08: 1451 tiles disponibles
  2023-06: 900 tiles disponibles
  2023-07: 718 tiles disponibles
  2023-08: 1043 tiles disponibles
  2024-06: 646 tiles disponibles
  2024-07: 1384 tiles disponibles
  2024-08: 511 tiles disponibles
  2025-06: 1245 tiles disponibles
  2025-07: 985 tiles disponibles
  2025-08: 1003 tiles disponibles

Plan de muestreo (target 1000):
  2020-06: 90 de 2321 disponibles
  2020-07: 56 de 1464 disponibles
  2020-08: 74 de 1907 disponibles
  2021-06:

Procesando pendientes: 100%|██████████| 81/81 [07:20<00:00,  5.44s/it]


¡GOLDEN DATASET LISTO! -> D:\Silos\Golden_Dataset_1000
Total imágenes: 615 (534 previas + 81 nuevas)
Log de selección: selection_log.txt


In [5]:
import os
import shutil
from tqdm import tqdm

# --- CONFIGURACIÓN ---
GOLDEN_DIR = r"D:\Silos\Golden_Dataset_1000" # Donde ya están tus imágenes y predicciones
SOURCE_SHP_DIR = r"D:\Silos\Base de datos\procesado_sin_nubes\tiles\indices_INBNV\masks\polygons" # Donde buscar los GT

def fix_missing_gt():
    print(f"--- REPARANDO DATASET EN: {GOLDEN_DIR} ---")
    
    # 1. Listar las imágenes TIF que ya están en la carpeta Golden
    tif_files = [f for f in os.listdir(GOLDEN_DIR) if f.lower().endswith('.tif')]
    
    if not tif_files:
        print("Error: No encontré imágenes .tif en la carpeta Golden.")
        return

    print(f"Buscando GT para {len(tif_files)} imágenes...")
    
    found_count = 0
    missing_count = 0
    
    # 2. Iterar y buscar su pareja
    for tif_name in tqdm(tif_files):
        base_name = os.path.splitext(tif_name)[0]
        
        # El nombre del archivo GT destino
        dst_gt_base = os.path.join(GOLDEN_DIR, f"gt_{base_name}")
        
        # Buscar el archivo origen. Intentamos coincidencia exacta primero.
        # Asumimos que el shapefile se llama igual que la imagen.
        
        # Lista de extensiones necesarias para un shapefile
        extensions = ['.shp', '.shx', '.dbf', '.prj', '.cpg']
        
        # Verificamos si existe el .shp principal primero
        src_shp_main = os.path.join(SOURCE_SHP_DIR, base_name + ".shp")
        
        if os.path.exists(src_shp_main):
            found_count += 1
            # Copiar todas las extensiones asociadas
            for ext in extensions:
                src_file = os.path.join(SOURCE_SHP_DIR, base_name + ext)
                dst_file = dst_gt_base + ext
                
                if os.path.exists(src_file):
                    shutil.copy2(src_file, dst_file)
        else:
            # INTENTO DE RECUPERACIÓN (Búsqueda laxa)
            # A veces el tif se llama "imagen_01.tif" y el shape "imagen_01_poly.shp" o similar.
            # Buscamos en la carpeta origen algún archivo que empiece igual.
            candidates = [f for f in os.listdir(SOURCE_SHP_DIR) if f.startswith(base_name) and f.endswith(".shp")]
            
            if candidates:
                # Tomamos el primero que encontremos (usualmente es el correcto)
                best_match = candidates[0]
                match_base = os.path.splitext(best_match)[0]
                found_count += 1
                
                for ext in extensions:
                    src_file = os.path.join(SOURCE_SHP_DIR, match_base + ext)
                    dst_file = dst_gt_base + ext # Lo guardamos con el nombre LIMPIO (igual al tif)
                    
                    if os.path.exists(src_file):
                        shutil.copy2(src_file, dst_file)
            else:
                missing_count += 1
                # print(f"Falta GT para: {base_name}") # Descomentar si quieres ver cuáles faltan

    print("\n" + "="*40)
    print(f"Resumen de Reparación:")
    print(f"✅ GTs encontrados y copiados: {found_count}")
    print(f"❌ GTs no encontrados (quizás no existen): {missing_count}")
    print("="*40)
    print("Ahora revisa tu carpeta Golden, deberían aparecer los archivos 'gt_...'.")

if __name__ == "__main__":
    fix_missing_gt()

--- REPARANDO DATASET EN: D:\Silos\Golden_Dataset_1000 ---
Buscando GT para 615 imágenes...


  0%|          | 0/615 [00:00<?, ?it/s]

100%|██████████| 615/615 [02:36<00:00,  3.93it/s]


Resumen de Reparación:
✅ GTs encontrados y copiados: 615
❌ GTs no encontrados (quizás no existen): 0
Ahora revisa tu carpeta Golden, deberían aparecer los archivos 'gt_...'.


In [6]:
import os
import re
import pandas as pd

# --- CONFIGURACIÓN ---
GOLDEN_DIR = r"D:\Silos\Golden_Dataset_1000"
OUTPUT_EXCEL = r"D:\Silos\Golden_Dataset_1000\Planilla_Seguimiento_Golden.xlsx"

def generar_excel():
    print(f"Escaneando carpeta: {GOLDEN_DIR} ...")
    
    if not os.path.exists(GOLDEN_DIR):
        print("Error: No existe la carpeta del Golden Dataset.")
        return

    archivos = sorted([f for f in os.listdir(GOLDEN_DIR) if f.lower().endswith('.tif')])
    print(f"Se encontraron {len(archivos)} imágenes.")

    # Parsear año y mes del nombre
    years = []
    months = []
    for f in archivos:
        m = re.match(r'(\d{4})_(\d{1,2})_', f)
        if m:
            years.append(int(m.group(1)))
            months.append(int(m.group(2)))
        else:
            years.append(None)
            months.append(None)

    df = pd.DataFrame({
        'ID_Imagen': archivos,
        'Año': years,
        'Mes': months,
        'Estado': 'Pendiente',
        '¿Requiere Corrección?': 'No',
        'Silos_GT_Eliminados': 0,      # Cuántos polígonos falsos borraste
        'Silos_Nuevos_Agregados': 0,   # Cuántos silos nuevos marcaste a mano
        'Dificultad': 'Baja',
        'Comentarios': ''
    })

    try:
        df.to_excel(OUTPUT_EXCEL, index=False)
        print(f"\n✅ Excel guardado en: {OUTPUT_EXCEL}")
        print(f"   Total imágenes: {len(archivos)}")
        print(f"   Años: {sorted(set(y for y in years if y))}")
        print(f"   Meses: {sorted(set(m for m in months if m))}")
    except Exception as e:
        print(f"Error al guardar: {e}")

if __name__ == "__main__":
    generar_excel()

Escaneando carpeta: D:\Silos\Golden_Dataset_1000 ...
Se encontraron 615 imágenes.

✅ Excel guardado en: D:\Silos\Golden_Dataset_1000\Planilla_Seguimiento_Golden.xlsx
   Total imágenes: 615
   Años: [2020, 2021, 2022, 2023, 2024, 2025]
   Meses: [6, 7, 8]
